# Demostraciones visuales de los conceptos

Notebook **didáctico**. No forma parte del proceso: no lee datos, no entrena nada y no escribe
ninguna tabla. Por eso no lleva número.

Reúne las representaciones visuales de **los conceptos** sobre los que se apoya el proyecto.
En el flujo de trabajo estos conceptos se aplican sin detenerse a dibujarlos, porque allí lo que
importa es el resultado. Aquí cada uno tiene su figura, y cada figura responde a una pregunta
concreta.

| Figura | Responde a |
|---|---|
| C1 | ¿Por qué una regresión logística no puede usar una recta? |
| C2 | ¿Por qué `age_group` no es cosmética? |
| C3 | ¿Qué miden exactamente Gini y la entropía? |
| C4 | ¿Qué hace un árbol que una recta no puede hacer? |
| C5 | ¿Por qué un bosque es mejor que un árbol? |
| C6 | ¿Por qué decide la average precision y no el ROC-AUC? |
| C7 | ¿Por qué una distribución uniforme no tiene valores atípicos? |
| C8 | ¿Qué significa que un modelo esté calibrado? |
| C9 | ¿Qué ocurre dentro del preprocesado? |
| C10 | ¿Dónde vive el conjunto de prueba y por qué no se toca? |

Las cifras que aparecen son las del proyecto y están escritas a mano a propósito: así este
notebook se puede ejecutar en cualquier sitio, sin conexión a la base de datos.


In [ ]:
# Se ejecuta en cualquier entorno con matplotlib y numpy. No necesita Spark ni datos.
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

# Misma paleta que el resto del proyecto y que la documentación
CHURN, SAFE, ACC, GREY = "#a8471f", "#2c5a72", "#8a5a08", "#78736a"
INK, PAPER = "#1c1a17", "#efe9dd"
CAJA = dict(boxstyle="round,pad=0.28", facecolor="white", edgecolor="none", alpha=0.85)

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 160, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.22,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
})

# Cifras del proyecto, escritas a mano para que el notebook sea independiente
TASA_BASE   = 0.2037
UMBRAL      = 0.194
TRAMOS      = ["18-29", "30-39", "40-49", "50-59", "60+"]
EDAD_PCT    = [7.6, 10.9, 30.8, 56.0, 27.9]
ACTIVOS     = [5.4, 8.2, 22.6, 37.1, 12.5]
INACTIVOS   = [9.8, 13.6, 38.0, 81.2, 85.6]

def caja_bigotes(ax, datos, horizontal=False, **kw):
    """Diagrama de caja compatible con cualquier versión de matplotlib.

    En la 3.10 el argumento `vert` pasó a estar en desuso y se sustituyó por
    `orientation`. En versiones más nuevas ya no existe, así que llamarlo
    directamente rompe la celda. Se prueba primero el nuevo y se cae al viejo.
    """
    try:
        return ax.boxplot(datos, orientation="horizontal" if horizontal else "vertical", **kw)
    except TypeError:
        return ax.boxplot(datos, vert=not horizontal, **kw)


import matplotlib
print("matplotlib:", matplotlib.__version__)
print("paleta y constantes listas")

## Dónde se guardan las figuras

Se intenta un volumen de Unity Catalog, que es lo que permite descargarlas después. Si no
está disponible, se cae al disco local del driver y se avisa.


In [ ]:
def preparar_destino():
    """Devuelve la carpeta donde guardar los PNG, probando volumen y luego disco local."""
    try:
        spark.sql("CREATE VOLUME IF NOT EXISTS bank_churn.gold.figuras")
        destino = "/Volumes/bank_churn/gold/figuras"
        os.makedirs(destino, exist_ok=True)
        print(f"destino: {destino}  (volumen de Unity Catalog)")
        print("Para descargarlas: Catalog > bank_churn > gold > Volumes > figuras")
        return destino
    except Exception as e:
        destino = "/tmp/figuras"
        os.makedirs(destino, exist_ok=True)
        print(f"[aviso] no se pudo usar un volumen ({type(e).__name__}). Se usa {destino}")
        print("Las figuras quedan igualmente incrustadas en la salida del notebook.")
        return destino


DESTINO = preparar_destino()


def guardar(fig, nombre):
    """Guarda la figura y confirma la ruta."""
    ruta = os.path.join(DESTINO, f"{nombre}.png")
    fig.savefig(ruta)
    print(f"  guardada: {ruta}")
    return ruta

---

## C1 · Por qué la logística no puede usar una recta

Una probabilidad vive entre 0 y 1. Una recta se sale por los dos lados: predice un 130 % y un
−20 %, que no significan nada. La sigmoide resuelve eso aplastando cualquier número real al
intervalo abierto (0, 1).

**Lo que hay que leer en la figura:** la recta cruza el 1 y el 0 y sigue; la sigmoide se acerca
sin llegar nunca. Y el umbral de rentabilidad del proyecto, `p = 0,194`, corresponde a un
log-odds de −1,42: la frontera del negocio no está en el centro, está claramente desplazada
hacia el «no».


In [ ]:
z = np.linspace(-6, 6, 400)
sig = 1 / (1 + np.exp(-z))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))

# --- izquierda: el problema ---
ax1.plot(z, 0.5 + 0.13 * z, color=GREY, lw=2.4, label="una recta")
ax1.axhspan(1, 1.5, color=CHURN, alpha=0.10)
ax1.axhspan(-0.5, 0, color=CHURN, alpha=0.10)
ax1.axhline(1, color=CHURN, ls="--", lw=1.2)
ax1.axhline(0, color=CHURN, ls="--", lw=1.2)
ax1.text(-5.6, 1.18, "probabilidad > 1: no significa nada", color=CHURN, fontsize=9)
ax1.text(-5.6, -0.32, "probabilidad < 0: tampoco", color=CHURN, fontsize=9)
ax1.set_ylim(-0.5, 1.5); ax1.set_xlim(-6, 6)
ax1.set_title("El problema: una recta se sale", loc="left")
ax1.set_xlabel("combinación lineal de las variables (z)"); ax1.set_ylabel("probabilidad")
ax1.legend(loc="lower right", fontsize=9)

# --- derecha: la solución ---
ax2.plot(z, sig, color=CHURN, lw=2.8, label=r"sigmoide:  p = 1 / (1 + e$^{-z}$)")
ax2.axhline(1, color=GREY, ls=":", lw=1); ax2.axhline(0, color=GREY, ls=":", lw=1)

for zz, etiqueta, color in [(0.0, "z = 0  ->  p = 0,50", SAFE),
                            (-1.42, "z = −1,42  ->  p = 0,194", CHURN)]:
    pp = 1 / (1 + np.exp(-zz))
    ax2.plot([zz, zz], [0, pp], color=color, ls="--", lw=1.2)
    ax2.plot([-6, zz], [pp, pp], color=color, ls="--", lw=1.2)
    ax2.plot(zz, pp, "o", color=color, ms=9, zorder=5)
    ax2.text(zz + 0.30, pp - 0.055 if zz < 0 else pp + 0.05, etiqueta,
             color=color, fontsize=9, bbox=CAJA, zorder=6)

ax2.text(-5.7, 0.70, "el umbral de rentabilidad del proyecto\nno está en el centro (z = 0), sino\nclaramente desplazado hacia el «no»",
         fontsize=8.5, color=INK, bbox=CAJA, va="top")
ax2.set_ylim(-0.08, 1.12); ax2.set_xlim(-6, 6)
ax2.set_title("La solución: la sigmoide", loc="left")
ax2.set_xlabel("log-odds (z)"); ax2.set_ylabel("probabilidad")
ax2.legend(loc="upper left", fontsize=9)

fig.suptitle("Figura C1 · Por qué la regresión logística modela el log-odds y no la probabilidad",
             fontsize=12.5, fontweight="bold", x=0.008, ha="left")
plt.tight_layout(); guardar(fig, "C1_sigmoide"); plt.show()

---

## C2 · Por qué `age_group` no es cosmética

Este es el argumento que justifica una de las decisiones de diseño del proyecto: la regresión
logística recibe la edad troceada en cinco tramos, y los árboles la reciben en crudo.

**Lo que hay que leer:** la recta que mejor se ajusta a los cinco puntos falla estrepitosamente en
el tramo de más de 60 años. Le asignaría un riesgo alto a un grupo que en realidad ha vuelto al
nivel de un cliente de 40. Con cinco cajas, cada tramo puede tener su propio coeficiente y el
problema desaparece.


In [ ]:
x = np.arange(5)
fig, ax = plt.subplots(figsize=(9.5, 4.8))

colores = [CHURN if v >= TASA_BASE * 100 else SAFE for v in EDAD_PCT]
ax.bar(x, EDAD_PCT, color=colores, width=0.6, zorder=2, label="tasa real por tramo")

# la recta que ajustaría un modelo lineal con la edad en crudo
m, b = np.polyfit(x, EDAD_PCT, 1)
ax.plot(x, m * x + b, color=GREY, lw=2.6, ls="--", zorder=4,
        label="lo que ajustaría una recta")

# el error en el ultimo tramo
pred_60 = m * 4 + b
ax.annotate("", xy=(4, EDAD_PCT[4]), xytext=(4, pred_60),
            arrowprops=dict(arrowstyle="<->", color=CHURN, lw=2))
ax.text(3.55, (EDAD_PCT[4] + pred_60) / 2,
        f"error de {pred_60 - EDAD_PCT[4]:.0f} puntos\nen el tramo 60+",
        color=INK, fontsize=9, ha="right", va="center", bbox=CAJA, zorder=6)

for i, v in enumerate(EDAD_PCT):
    ax.text(i, v + 1.4, f"{v}%", ha="center", fontweight="bold", fontsize=9.5, color=INK, zorder=5)

ax.set_xticks(x); ax.set_xticklabels(TRAMOS)
ax.set_ylabel("% de abandono"); ax.set_xlabel("tramo de edad")
ax.set_ylim(0, 70)
ax.set_title("Figura C2 · Una recta no puede representar una U invertida", loc="left")
ax.legend(fontsize=9.5, loc="upper left")
plt.tight_layout(); guardar(fig, "C2_u_invertida_vs_recta"); plt.show()

---

## C3 · Qué miden Gini y la entropía

Las dos responden a la misma pregunta: **¿cuánto se parece este grupo a una moneda al aire?**
Un grupo puro vale cero en ambas; uno al 50/50 vale el máximo.

**Lo que hay que leer:** las dos curvas tienen la misma forma y el mismo máximo en 0,5. Por eso
en la práctica dan casi siempre el mismo árbol. Y el punto marcado es nuestro nodo raíz: con un
20,37 % de abandono, ya se parte de un grupo bastante desequilibrado, lo cual deja margen para
que las particiones mejoren.


In [ ]:
p = np.linspace(0.001, 0.999, 400)
gini = 1 - (p**2 + (1 - p)**2)
entr = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.plot(p, gini, color=CHURN, lw=2.8, label="Gini = 1 − Σ pᵢ²   (máximo 0,5)")
ax.plot(p, entr, color=SAFE,  lw=2.8, label="Entropía = − Σ pᵢ log₂ pᵢ   (máximo 1 bit)")

g0 = 1 - (TASA_BASE**2 + (1 - TASA_BASE)**2)
e0 = -(TASA_BASE * np.log2(TASA_BASE) + (1 - TASA_BASE) * np.log2(1 - TASA_BASE))
ax.axvline(TASA_BASE, color=GREY, ls="--", lw=1.3)
ax.plot([TASA_BASE, TASA_BASE], [g0, e0], "o", color=INK, ms=8, zorder=5)
ax.text(TASA_BASE + 0.025, e0 + 0.03,
        f"nuestro nodo raíz\np = {TASA_BASE:.4f}\nGini {g0:.3f} · entropía {e0:.3f} bits",
        fontsize=9, color=INK, bbox=CAJA, zorder=6)

ax.text(0.5, 1.06, "máxima incertidumbre:\nuna moneda al aire", ha="center", fontsize=8.5,
        color=GREY, bbox=CAJA)
ax.text(0.955, 0.08, "grupo puro:\nno hace falta\npreguntar más", ha="right", fontsize=8.5,
        color=GREY, bbox=CAJA)

ax.set_xlabel("proporción de la clase positiva en el nodo")
ax.set_ylabel("impureza"); ax.set_ylim(0, 1.2); ax.set_xlim(0, 1)
ax.set_title("Figura C3 · Las dos formas de medir la impureza de un nodo", loc="left")
ax.legend(fontsize=9.5, loc="upper right")
plt.tight_layout(); guardar(fig, "C3_gini_entropia"); plt.show()

---

## C4 · Qué hace un árbol que una recta no puede hacer

El hallazgo principal del proyecto es una **interacción**: el efecto de la edad depende de la
actividad. Esta figura muestra por qué un modelo lineal no puede capturarla y un árbol sí.

**Lo que hay que leer:** a la izquierda, una recta solo puede separar el plano en dos mitades.
A la derecha, el árbol lo trocea en rectángulos y puede darle a cada uno su propio riesgo. El
camino «edad > 42 **y** además inactivo» **es** la interacción, y el árbol la construye sola.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.9))

rng = np.random.default_rng(42)
edad = rng.uniform(18, 92, 420)
activo = rng.integers(0, 2, 420)
# riesgo segun el hallazgo real: entre inactivos crece sin parar
riesgo = np.where(activo == 1,
                  np.clip(0.05 + 0.010 * (edad - 18) - 0.00022 * (edad - 18) ** 2, 0, 1),
                  np.clip(0.06 + 0.014 * (edad - 18), 0, 1))
abandona = rng.random(420) < riesgo
y_j = activo + rng.normal(0, 0.055, 420)

for ax, titulo in [(ax1, "Un modelo lineal: una sola frontera recta"),
                   (ax2, "Un árbol: el plano troceado en rectángulos")]:
    ax.scatter(edad[~abandona], y_j[~abandona], s=17, color=SAFE,  alpha=0.55, label="se queda")
    ax.scatter(edad[abandona],  y_j[abandona],  s=17, color=CHURN, alpha=0.75, label="abandona")
    ax.set_yticks([0, 1]); ax.set_yticklabels(["inactivo", "activo"])
    ax.set_xlabel("edad"); ax.set_xlim(16, 94); ax.set_ylim(-0.45, 1.45)
    ax.set_title(titulo, loc="left", fontsize=11.5)

ax1.plot([16, 94], [1.35, -0.35], color=GREY, lw=2.8, ls="--")
ax1.text(20, -0.33, "una recta no puede decir\n«mayor Y ADEMÁS inactivo»",
         fontsize=9, color=INK, bbox=CAJA)

for x0, x1, y0, y1, c, txt in [
        (16, 42, -0.45, 0.5, SAFE,  "riesgo bajo"),
        (42, 94, -0.45, 0.5, CHURN, "riesgo MUY alto"),
        (16, 42,  0.5, 1.45, SAFE,  "riesgo bajo"),
        (42, 94,  0.5, 1.45, ACC,   "riesgo medio")]:
    ax2.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor=c, alpha=0.13, zorder=0))
    ax2.text((x0 + x1) / 2, y1 - 0.13, txt, ha="center", fontsize=8.5, color=INK, bbox=CAJA)
ax2.axvline(42, color=INK, lw=2)
ax2.axhline(0.5, color=INK, lw=2)
ax2.text(43, -0.38, "corte 1: ¿edad > 42?", fontsize=8.5, color=INK, bbox=CAJA)

ax1.legend(fontsize=9, loc="upper left")
fig.suptitle("Figura C4 · Por qué la interacción edad × actividad necesita un árbol",
             fontsize=12.5, fontweight="bold", x=0.008, ha="left")
plt.tight_layout(); guardar(fig, "C4_arbol_vs_recta"); plt.show()

---

## C5 · Por qué un bosque es mejor que un árbol

Un árbol solo es **inestable**: cambiar unas pocas filas de entrenamiento puede cambiar el corte
de la raíz y con él todo lo de abajo. El bosque entrena muchos y promedia.

**Lo que hay que leer:** cada árbol ve un conjunto distinto de filas (bagging) y, en cada corte,
solo puede mirar algunas columnas (`max_features`). Esa segunda restricción es la imprescindible:
sin ella todos empezarían partiendo por la edad y se parecerían demasiado, y **los errores
correlacionados no se cancelan al promediar**.


In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.2))
ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 6.6)

# origen
ax.add_patch(Rectangle((0.2, 2.6), 1.5, 1.2, facecolor=PAPER, edgecolor=INK, lw=1.4))
ax.text(0.95, 3.2, "8.000\nfilas", ha="center", va="center", fontsize=10, fontweight="bold")

muestras = [
    (4.45, "muestra 1\n63 % de las filas", "columnas: edad, saldo, país…"),
    (2.95, "muestra 2\notras filas", "columnas: productos, actividad…"),
    (1.05, "muestra 600\notras filas", "columnas: saldo, antigüedad…"),
]
for y, titulo, cols in muestras:
    ax.add_patch(FancyArrowPatch((1.8, 3.2), (3.0, y + 0.45), arrowstyle="->",
                                 mutation_scale=14, color=GREY, lw=1.3))
    ax.add_patch(Rectangle((3.1, y), 2.6, 0.9, facecolor="white", edgecolor=CHURN, lw=1.3))
    ax.text(4.4, y + 0.62, titulo.split(chr(10))[0], ha="center", fontsize=9, fontweight="bold")
    ax.text(4.4, y + 0.24, titulo.split(chr(10))[1], ha="center", fontsize=8, color=GREY)
    ax.text(6.0, y + 0.45, cols, fontsize=8, color=SAFE, va="center")
    ax.add_patch(FancyArrowPatch((8.35, y + 0.45), (9.0, 3.2), arrowstyle="->",
                                 mutation_scale=14, color=GREY, lw=1.3))

ax.text(4.4, 2.35, "⋮", ha="center", fontsize=20, color=GREY)
ax.add_patch(Rectangle((9.05, 2.6), 0.85, 1.2, facecolor=PAPER, edgecolor=CHURN, lw=1.8))
ax.text(9.47, 3.2, "media", ha="center", va="center", fontsize=9.5, fontweight="bold")

ax.text(0.2, 6.35, "Figura C5 · De un árbol al bosque: la diversidad se fabrica a propósito",
        fontsize=12.5, fontweight="bold")
ax.text(0.2, 5.95, "Dos fuentes de aleatoriedad: cada árbol ve filas distintas (bagging) y, en cada corte, solo algunas columnas (max_features).",
        fontsize=9.5, color=GREY)
ax.text(3.1, 0.35, "Sin la segunda, todos los árboles empezarían partiendo por la edad y sus errores no se cancelarían al promediar.",
        fontsize=9, color=INK, bbox=CAJA)
guardar(fig, "C5_arbol_a_bosque"); plt.show()

---

## C6 · Por qué decide la average precision y no el ROC-AUC

Las dos curvas miden lo mismo desde ángulos distintos, y con clases desbalanceadas no dan la
misma impresión.

**Lo que hay que leer:** el suelo de la curva ROC es siempre 0,5, gane quien gane. El suelo de la
curva precisión-recall es **la tasa base**, que en nuestro caso es 0,2037 — y es exactamente lo
que obtuvo el clasificador trivial. Eso convierte la métrica en legible: 0,2037 es no saber nada,
1,0 sería perfecto, y nuestro 0,6848 está algo más allá de la mitad del camino.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.7))

r = np.linspace(0.001, 1, 300)
# curva PR ilustrativa con la forma de un modelo como el nuestro
prec = np.clip(0.95 * np.exp(-2.1 * r) + TASA_BASE, TASA_BASE, 1)

ax1.plot(r, prec, color=CHURN, lw=2.8, label="nuestro modelo")
ax1.axhline(TASA_BASE, color=GREY, ls="--", lw=1.6)
ax1.fill_between(r, TASA_BASE, prec, color=CHURN, alpha=0.10)
ax1.text(0.5, TASA_BASE - 0.055, f"suelo del azar = tasa base = {TASA_BASE:.4f}",
         fontsize=9, color=INK, ha="center", bbox=CAJA)
ax1.text(0.42, 0.62, "el área sombreada\nes la average precision", fontsize=9, color=INK, bbox=CAJA)
ax1.set_xlabel("recall (cobertura)"); ax1.set_ylabel("precisión")
ax1.set_ylim(0, 1.05); ax1.set_xlim(0, 1)
ax1.set_title("Curva precisión-recall: el suelo depende de los datos", loc="left", fontsize=11.5)

fpr = np.linspace(0, 1, 300)
tpr = fpr ** 0.32
ax2.plot(fpr, tpr, color=SAFE, lw=2.8, label="nuestro modelo")
ax2.plot([0, 1], [0, 1], color=GREY, ls="--", lw=1.6)
ax2.fill_between(fpr, fpr, tpr, color=SAFE, alpha=0.10)
ax2.text(0.58, 0.42, "suelo del azar = 0,5\nsiempre, pase lo que pase",
         fontsize=9, color=INK, bbox=CAJA)
ax2.set_xlabel("tasa de falsos positivos"); ax2.set_ylabel("tasa de verdaderos positivos")
ax2.set_ylim(0, 1.05); ax2.set_xlim(0, 1)
ax2.set_title("Curva ROC: el suelo es fijo", loc="left", fontsize=11.5)

fig.suptitle("Figura C6 · Con clases desbalanceadas, solo una de las dos tiene un suelo interpretable",
             fontsize=12.5, fontweight="bold", x=0.008, ha="left")
plt.tight_layout(); guardar(fig, "C6_ap_vs_roc"); plt.show()

---

## C7 · Por qué una distribución uniforme no tiene valores atípicos

Este es el argumento que cierra la demostración de que el conjunto de datos es sintético, y no
es una impresión visual: es aritmética.

**Lo que hay que leer:** en una uniforme, los cuartiles caen al 25 % y al 75 % del recorrido, así
que el rango intercuartílico es la mitad del rango total y las vallas se van muy por fuera de los
datos. **El diagrama de caja no puede marcar ni un solo punto, garantizado por construcción.**
En una normal caería fuera alrededor del 0,7 %.

La frase que debe quedar: *un diagrama de caja completamente limpio no significa que la variable
esté sana; es la firma de una distribución plana. La ausencia de atípicos es la prueba, no la
tranquilidad.*


In [ ]:
rng = np.random.default_rng(7)
n = 10000
unif = rng.uniform(0, 200000, n)          # como el salario estimado del proyecto
norm = rng.normal(100000, 33000, n)       # una variable con centro, para comparar

fig, axes = plt.subplots(2, 2, figsize=(12.5, 6.6),
                         gridspec_kw={"height_ratios": [1, 1.5]})

for j, (datos, nombre, color) in enumerate([(unif, "uniforme (como el salario estimado)", CHURN),
                                            (norm, "con centro (una normal)", SAFE)]):
    bp = caja_bigotes(axes[0, j], datos, horizontal=True, widths=0.55, patch_artist=True,
                      flierprops=dict(marker="o", ms=3, mfc=CHURN, mec="none", alpha=0.5))
    bp["boxes"][0].set(facecolor=color, alpha=0.35)
    for k in ("whiskers", "caps", "medians"):
        for e in bp[k]: e.set(color=INK, lw=1.3)
    n_out = len(bp["fliers"][0].get_xdata())   # horizontal: los atipicos van en X
    axes[0, j].set_yticks([])
    axes[0, j].set_title(f"{nombre}   ->   {n_out} atípicos marcados",
                         loc="left", fontsize=11)
    axes[0, j].grid(False)

    axes[1, j].hist(datos, bins=55, color=color, alpha=0.75, edgecolor="white", lw=0.4)
    axes[1, j].set_xlabel("valor"); axes[1, j].set_ylabel("clientes")

q1, q3 = np.percentile(unif, [25, 75]); iqr = q3 - q1
axes[1, 0].text(0.03, 0.93,
    f"Q1 = {q1:,.0f}\nQ3 = {q3:,.0f}\nIQR = {iqr:,.0f}  (la mitad del rango)\n"
    f"valla inferior = {q1 - 1.5*iqr:,.0f}\nvalla superior = {q3 + 1.5*iqr:,.0f}\n"
    f"datos reales = [{unif.min():,.0f} · {unif.max():,.0f}]",
    transform=axes[1, 0].transAxes, va="top", fontsize=8.5, family="monospace",
    color=INK, bbox=CAJA)
axes[1, 0].text(0.5, 0.45, "plano como una mesa", transform=axes[1, 0].transAxes,
                ha="center", fontsize=10, color=INK, bbox=CAJA)

fig.suptitle("Figura C7 · Un diagrama de caja limpio no es tranquilidad: es la firma de una distribución plana",
             fontsize=12.5, fontweight="bold", x=0.008, ha="left")
plt.tight_layout(); guardar(fig, "C7_atipicos_uniforme"); plt.show()

---

## C8 · Qué significa que un modelo esté calibrado

Discriminar y calibrar son propiedades **independientes**. Un modelo puede ordenar perfectamente
a los clientes y aun así mentir en los números que muestra.

**Lo que hay que leer:** en la diagonal, lo que el modelo dice coincide con lo que ocurre. La
curva de abajo es la del modelo sin calibrar: donde decía 49 %, la realidad era 19,5 %.

**Y por qué no es cosmético:** calibrar no cambia a quién se contacta, porque no altera el orden.
Cambia el número que lee el equipo de retención, y es lo que permite repartir el cupo por país
sumando probabilidades.


In [ ]:
pred = np.linspace(0.02, 0.98, 11)
real_mal  = pred * 0.42          # sobreestima, como el modelo sin calibrar
real_bien = pred + np.array([0.02, -0.01, 0.03, -0.02, 0.01, 0.02, -0.03, 0.01, -0.01, 0.02, -0.01])

fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.plot([0, 1], [0, 1], color=GREY, ls="--", lw=1.8, label="calibración perfecta")
ax.plot(pred, real_mal,  "o-", color=CHURN, lw=2.4, ms=7, label="sin calibrar: sobreestima siempre")
ax.plot(pred, real_bien, "o-", color=SAFE,  lw=2.4, ms=7, label="tras la calibración isotónica")

i = 5
ax.annotate("", xy=(pred[i], real_mal[i]), xytext=(pred[i], pred[i]),
            arrowprops=dict(arrowstyle="<->", color=CHURN, lw=1.8))
ax.text(pred[i] + 0.03, (pred[i] + real_mal[i]) / 2,
        "donde el modelo decía 49 %,\nla realidad era 19,5 %\n(29,7 puntos de desvío)",
        fontsize=9, color=INK, va="center", bbox=CAJA)

ax.set_xlabel("probabilidad que predice el modelo")
ax.set_ylabel("proporción real de abandonos")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Figura C8 · Discriminar bien y calibrar bien son cosas distintas", loc="left")
ax.legend(fontsize=9.5, loc="upper left")
plt.tight_layout(); guardar(fig, "C8_calibracion"); plt.show()

---

## C9 · Qué ocurre dentro del preprocesado

Trece variables entran y salen dieciséis columnas por un lado y doce por el otro. Esta figura
descompone de dónde sale cada número.

**Lo que hay que leer:** no es duplicar trabajo, es no penalizar a ninguna de las dos familias.
La logística necesita la edad troceada y escalada; el árbol la prefiere en crudo, donde tiene
hasta 74 puntos de corte en vez de cinco cajas fijas.


In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 6))
ax.axis("off"); ax.set_xlim(0, 12); ax.set_ylim(0, 9.1)

ax.add_patch(Rectangle((0.15, 3.4), 1.9, 1.8, facecolor=PAPER, edgecolor=INK, lw=1.5))
ax.text(1.1, 4.5, "13", ha="center", fontsize=20, fontweight="bold", color=CHURN)
ax.text(1.1, 3.85, "variables\ncandidatas", ha="center", fontsize=9)

ramas = [
    (4.9, "prep_lineal", "16", CHURN, [
        "geography (3 niveles)  ->  2",
        "gender (2)  ->  1",
        "age_group (5)  ->  4",
        "products_group (3)  ->  2",
        "4 numéricas escaladas  ->  4",
        "3 binarias tal cual  ->  3"], "para la REGRESIÓN LOGÍSTICA"),
    (0.5, "prep_arbol", "12", SAFE, [
        "geography (3)  ->  2",
        "gender (2)  ->  1",
        "6 numéricas en crudo  ->  6",
        "   (incluye age y num_of_products)",
        "3 binarias tal cual  ->  3",
        "age_group y products_group: DESCARTADAS"], "para el ÁRBOL y el BOSQUE"),
]
for y, nombre, total, color, lineas, destino in ramas:
    ax.add_patch(FancyArrowPatch((2.15, 4.3), (3.15, y + 1.3), arrowstyle="->",
                                 mutation_scale=16, color=GREY, lw=1.5))
    ax.add_patch(Rectangle((3.2, y), 5.4, 2.6, facecolor="white", edgecolor=color, lw=1.6))
    ax.text(3.45, y + 2.25, nombre, fontsize=10.5, fontweight="bold",
            color=color, family="monospace")
    for k, ln in enumerate(lineas):
        ax.text(3.45, y + 1.85 - k * 0.31, ln, fontsize=8.3, family="monospace", color=INK)
    ax.add_patch(FancyArrowPatch((8.65, y + 1.3), (9.5, y + 1.3), arrowstyle="->",
                                 mutation_scale=16, color=GREY, lw=1.5))
    ax.add_patch(Rectangle((9.55, y + 0.75), 1.2, 1.1, facecolor=PAPER, edgecolor=color, lw=1.6))
    ax.text(10.15, y + 1.3, total, ha="center", va="center", fontsize=19,
            fontweight="bold", color=color)
    ax.text(10.15, y + 0.5, destino, ha="center", fontsize=7.8, color=GREY)

ax.text(0.15, 8.7, "Figura C9 · Trece variables, dos preprocesadores, dieciséis y doce columnas",
        fontsize=12.5, fontweight="bold")
ax.text(0.15, 8.25, "Cada familia recibe la forma que le conviene. Darles el mismo tratamiento penalizaría a una de las dos.",
        fontsize=9.5, color=GREY)
guardar(fig, "C9_preprocesado"); plt.show()

---

## C10 · Dónde vive el conjunto de prueba

La regla que gobierna todo el modelado: **el conjunto de prueba no se toca**. Esta figura muestra
dentro de qué recuadro se toma cada decisión.

**Lo que hay que leer:** todas las comparaciones, todos los ajustes y todas las mediciones de
importancia ocurren dentro del bloque de entrenamiento. Las 2.000 filas de la derecha están
selladas en una tabla y solo se abren una vez, en el notebook 05.


In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.6))
ax.axis("off"); ax.set_xlim(0, 12); ax.set_ylim(0, 7)

ax.add_patch(Rectangle((0.2, 5.6), 11.5, 0.85, facecolor=PAPER, edgecolor=INK, lw=1.5))
ax.text(5.95, 6.02, "10.000 clientes  ·  tasa base 20,37 %", ha="center", va="center",
        fontsize=11, fontweight="bold")

# entrenamiento
ax.add_patch(Rectangle((0.2, 1.0), 9.0, 4.1, facecolor="white", edgecolor=CHURN, lw=2))
ax.text(0.45, 4.75, "ENTRENAMIENTO · 8.000 filas", fontsize=10.5, fontweight="bold", color=CHURN)
ax.text(0.45, 4.38, "Aquí se toma TODA decisión del proyecto", fontsize=9, color=GREY)

for i in range(5):
    x = 0.5 + i * 1.72
    ax.add_patch(Rectangle((x, 3.0), 1.5, 1.0, facecolor=PAPER, edgecolor=GREY, lw=1))
    ax.add_patch(Rectangle((x, 3.0), 1.5, 1.0, facecolor=CHURN if i == 2 else "none",
                           alpha=0.22 if i == 2 else 0, edgecolor="none"))
    ax.text(x + 0.75, 3.5, f"partición {i+1}", ha="center", va="center", fontsize=8)
ax.text(0.5, 2.62, "Validación cruzada: se entrena con cuatro y se evalúa con la que queda, cinco veces.",
        fontsize=8.8, color=INK)

ax.add_patch(Rectangle((0.5, 1.35), 4.0, 0.95, facecolor=PAPER, edgecolor=SAFE, lw=1.2))
ax.text(2.5, 1.82, "partición interna del 25 %", ha="center", fontsize=9, fontweight="bold", color=SAFE)
ax.text(2.5, 1.52, "para la importancia por permutación", ha="center", fontsize=8, color=GREY)
ax.text(4.8, 1.82, "Se compara, se ajusta y se mide", fontsize=9, color=INK)
ax.text(4.8, 1.52, "SIN mirar nunca las 2.000 de la derecha", fontsize=9, color=INK, style="italic")

# prueba
ax.add_patch(Rectangle((9.5, 1.0), 2.2, 4.1, facecolor="#f5ddd9", edgecolor="#9a2d24", lw=2))
ax.text(10.6, 4.4, "PRUEBA", ha="center", fontsize=11, fontweight="bold", color="#9a2d24")
ax.text(10.6, 4.0, "2.000 filas", ha="center", fontsize=9.5, color="#9a2d24")
ax.text(10.6, 3.55, "407 abandonos", ha="center", fontsize=9, color=GREY)
ax.text(10.6, 2.55, "SELLADO", ha="center", fontsize=12, fontweight="bold", color="#9a2d24")
ax.text(10.6, 2.15, "silver.test_holdout", ha="center", fontsize=7.6, family="monospace", color=GREY)
ax.text(10.6, 1.45, "Se abre una sola vez,\nen el notebook 05", ha="center", fontsize=8.3, color=INK)

ax.text(0.2, 6.7, "Figura C10 · Cada decisión se toma dentro del recuadro naranja",
        fontsize=12.5, fontweight="bold")
ax.text(0.2, 0.45, "Cada vez que se mira el conjunto de prueba para decidir algo, se gasta: deja de estimar el rendimiento futuro y pasa a estimar «lo mejor que encontré mirando».",
        fontsize=9, color=INK, bbox=CAJA)
guardar(fig, "C10_test_holdout"); plt.show()

---

## Resumen

Diez figuras conceptuales, guardadas como PNG para reutilizarlas en el informe y en la
presentación sin tener que volver a ejecutar nada.

**Ninguna de estas figuras lee datos del proyecto**: las cifras que aparecen están escritas a mano
y coinciden con las del informe. Eso es deliberado, para que este notebook funcione aunque la base
de datos no esté disponible.

Las figuras que requieren los datos y los modelos entrenados están en `demo_figuras_proyecto`.
